# Module 09: Convolutions - Processing Images with Convolutions


In [57]:
#| default_exp foundation.convolutions
#| export

import numpy as np
rng = np.random.default_rng(7)
import time

from tinytorch.foundation.tensor import Tensor
from tinytorch.foundation.activations import ReLU
from tinytorch.foundation.layers import Linear

# Enable autograd for gradient tracking (required for BatchNorm2d learnable parameters)
from tinytorch.foundation.autograd import enable_autograd, Function, ReLUBackward
enable_autograd()

# Constants for convolution defaults
DEFAULT_KERNEL_SIZE = 3  # Default kernel size for convolutions
DEFAULT_STRIDE = 1  # Default stride for convolutions
DEFAULT_PADDING = 0  # Default padding for convolutions

# Constants for memory calculations
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
KB_TO_BYTES = 1024  # Kilobytes to bytes conversion
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion

## 🏗️ Implementation: Building Spatial Operations


### Conv2d: Detecting Patterns with Sliding Windows



### Shared Input Validation

All spatial operations (Conv2d, MaxPool2d, AvgPool2d) require 4D inputs shaped
as (batch, channels, height, width). Rather than duplicating this validation
logic three times, we define it once here.

In [2]:
#| export


def validate_4d_input(x, layer_name):
    """
    Validate that input tensor is 4D (batch, channels, height, width).
    """
    if len(x.shape) == 4:
        return  # Valid input

    if len(x.shape) == 3:
        raise ValueError(
            f"{layer_name} expected 4D input (batch, channels, height, width), got 3D: {x.shape}\n"
            f"  Missing batch dimension\n"
            f"  {layer_name} processes batches of images, not single images\n"
            f"  Add batch dim: x.reshape(1, {x.shape[0]}, {x.shape[1]}, {x.shape[2]})"
        )
    elif len(x.shape) == 2:
        raise ValueError(
            f"{layer_name} expected 4D input (batch, channels, height, width), got 2D: {x.shape}\n"
            f"  Got a matrix, expected an image tensor\n"
            f"  {layer_name} needs spatial dimensions (height, width) plus batch and channels\n"
            f"  If this is a flattened image, reshape it: x.reshape(1, channels, height, width)"
        )
    else:
        raise ValueError(
            f"{layer_name} expected 4D input (batch, channels, height, width), got {len(x.shape)}D: {x.shape}\n"
            f"  Wrong number of dimensions\n"
            f"  {layer_name} expects: (batch_size, channels, height, width)\n"
            f"  Reshape your input to 4D with the correct dimensions"
        )

### Conv2d Implementation - Building the Core of Computer Vision

Conv2d is the workhorse of computer vision. It slides learned filters across images to detect patterns like edges, textures, and eventually complex objects.

In [3]:
#| export

class Conv2dBackward(Function):
    """ Gradient computation for 2D convolution """

    def __init__(self, x, weight, bias, stride, padding, kernel_size):
        # Register all tensors that need gradient with autograd
        if bias is not None:
            super().__init__(x, weight, bias)
        else:
            super().__init__(x, weight)
        self.x = x
        self.weight = weight
        self.bias = bias
        self.stride = stride
        self.padding = padding
        self.kernel_size = kernel_size

    def apply(self, grad_outpout):
        """ Compute gradients for convolution inputs and parameters """

        batch_size, out_channels, out_height, out_width = grad_output.shape
        _, in_channels, in_height, in_width = self.x.shape
        kernel_h, kernel_w = slef.kernel_size

        # Apply padding to inputs if needed (for gradients computation)
        if self.padding > 0:
            padded_input = np.pad(self.x.data, 
                                 (0,0), (0,0), (self.padding,self.padding), (self.padding,self.padding),
                                 mode='constant', constant_values=0)
        else:
            padded_input = self.x.data

        # Initialize gradients
        grad_input_padded = np.zeros_like(padded_input)
        grad_weight = np.zeros_like(self.weight.data)
        grad_bias =  None if self.bias is None else np.zeros_like(self.bias.data)

        # Compute gradients using explicit loops
        for b in range(batch_size):
            for out_ch in range(out_channels):
                for out_h in range(out_height):
                    for out_w in range(out_width):
                        in_h_start = out_h * self.stride
                        in_w_start = in_w * self.stride

                        # Gradient value flowing back to this position
                        grad_val = grad_output(b, out_ch, out_h, out_w)
                        

                        
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                for in_ch in range(in_channels):
                                    # Input positions
                                    in_h = in_h_start + k_h
                                    in_w = in_w_start + k_w

                                    #Gradient w.r.t weight
                                    grad_weight[out_ch, in_ch, k_h, k_w] += (
                                    grad_val * padded_input[b, in_ch, in_h, in_w]
                                    )

                                    #Gradient w.r.t input
                                    grad_input_padded[b, in_ch, in_h, in_w] += (
                                    grad_val * self.weight.data[out_ch, in_ch, k_h, k_w] 
                                    )
        if grad_bias is not None:
            for out_ch in range(out_channels):
                grad_bias[out_ch] = grad_output[:, out_ch, :, :].sum()

        # Remove padding from input gradient
        if self.padding > 0:
            grad_input = grad_input_padded[:, :,
                                          self.padding:-self.padding,
                                          self.padding:-self.padding]
        else:
            grad_input = grad_input_padded

        if self.bias is None:
            return grad_input, grad_weight
        else:
            return grad_input, grad_weight, grad_bias
        
            

In [91]:
#| export

class Conv2d:
    """ 2D convolution layer for spatial features extraction. """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):

        self.in_channels = in_channels
        self.out_channels = out_channels

        # Handle kernel size as int or tuple
        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size

        self.stride = stride
        self.padding = padding   

        # He initialization for ReLU networks
        kernel_h, kernel_w = self.kernel_size
        fan_in = kernel_h * kernel_w * in_channels
        std = np.sqrt(2.0/fan_in)
        
        self.weight = Tensor(rng.normal(0, std, 
                                (out_channels, in_channels, kernel_h, kernel_w)),
                                requires_grad = True)

        # Bias initialization
        if bias:
            self.bias = Tensor(np.zeros(out_channels), requires_grad = True)
        else:
            self.bias = None

    def _compute_output_shape(self, in_h, in_w):
        """ Calculate output spatial dimensions for convolution """
        kernel_h, kernel_w = self.kernel_size
        out_w = (in_w + 2 * self.padding - kernel_w) // self.stride + 1
        out_h = (in_h + 2 * self.padding - kernel_h) // self.stride + 1

        return out_h, out_w

    def _apply_padding(self, x_data):
        """ Zero-pad the spatial input dimensions """

        if self.padding > 0:
            return np.pad(x_data,
                         ((0,0), (0,0), 
                         (self.padding,self.padding), (self.padding,self.padding)),
                          mode='constant',constant_values=0)
        else:
            return x_data

    def _convolve_loops(self, padded, batch_size, out_h, out_w):
        """ sliding window dot products over inputs. """

        out_channels = self.out_channels
        in_channels = self.in_channels
        kernel_h, kernel_w = self.kernel_size
        
        output = np.zeros((batch_size, out_channels, out_h, out_w))
        
        for b in range(batch_size):
            for out_c in range(self.out_channels):
                for oh in range(out_h):
                    for ow in range(out_w):
                        in_h_start = oh * self.stride
                        in_w_start = ow * self.stride
                        
                        conv_sum = 0.0
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                for in_c in range(in_channels):
                                    in_h = in_h_start + k_h
                                    in_w = in_w_start + k_w
                                    conv_sum += (self.weight.data[out_c, in_c, k_h, k_w] 
                                                * padded[b, in_c, in_h, in_w])
                        output[b, out_c, oh, ow] = conv_sum

        return output

    def forward(self, x):
        """ Forward pass through con2d layer """
        # Validate inputs
        validate_4d_input(x, "Conv2d")

        batch_size, in_channels, in_height, in_width = x.shape

        #Compute output dimensions
        out_height, out_width = self._compute_output_shape(in_height, in_width)

        # Apply padding
        padded_input = self._apply_padding(x.data)

        # Run convolution loop
        output = self._convolve_loops(padded_input, batch_size, out_height, out_width)

        # Add bias if present
        if self.bias is not None:
            for out_ch in range(self.out_channels):
                output[:, out_ch, :, :] = self.bias.data[out_ch]

        result = Tensor(output, requires_grad=(x.requires_grad or self.weight.requires_grad))

        if result.requires_grad:
            result._grad_fn = Conv2dBackward(
                x, self.weight, self.bias, self.stride,
                self.padding, self.kernel_size
            )
        return result

    def parameters(self):
        """Return trainable parameters."""
        params = [self.weight]
        if self.bias is not None:
            params.append(self.bias)
        return params

    def __call__(self, x):
        """Enable model(x) syntax."""
        return self.forward(x)
       
            
        
    

###  Unit Test: Conv2d Output Shape Computation

In [5]:


def test_unit_conv2d_output_shape():
    """Test Conv2d._compute_output_shape for various configurations."""
    print("Testing Conv2d output shape computation...")

    # Same padding: output == input
    conv_same = Conv2d(3, 16, kernel_size=3, padding=1, stride=1)
    oh, ow = conv_same._compute_output_shape(32, 32)
    assert (oh, ow) == (32, 32), f"Same padding: expected (32, 32), got ({oh}, {ow})"

    # No padding: output shrinks by (kernel - 1)
    conv_no_pad = Conv2d(3, 16, kernel_size=3, padding=0, stride=1)
    oh, ow = conv_no_pad._compute_output_shape(32, 32)
    assert (oh, ow) == (30, 30), f"No padding: expected (30, 30), got ({oh}, {ow})"

    # Stride 2: output roughly halves
    conv_stride = Conv2d(3, 16, kernel_size=3, padding=0, stride=2)
    oh, ow = conv_stride._compute_output_shape(32, 32)
    assert (oh, ow) == (15, 15), f"Stride 2: expected (15, 15), got ({oh}, {ow})"

    # Non-square input
    conv_rect = Conv2d(1, 8, kernel_size=3, padding=1, stride=1)
    oh, ow = conv_rect._compute_output_shape(28, 14)
    assert (oh, ow) == (28, 14), f"Rectangular: expected (28, 14), got ({oh}, {ow})"

    # Larger kernel
    conv_5x5 = Conv2d(3, 16, kernel_size=5, padding=0, stride=1)
    oh, ow = conv_5x5._compute_output_shape(32, 32)
    assert (oh, ow) == (28, 28), f"5x5 kernel: expected (28, 28), got ({oh}, {ow})"

    print("Conv2d output shape computation works correctly!")

if __name__ == "__main__":
    test_unit_conv2d_output_shape()

Testing Conv2d output shape computation...
Conv2d output shape computation works correctly!


###  Unit Test: Conv2d Padding

In [6]:
def test_unit_conv2d_padding():
    """Test Conv2d._apply_padding for zero-padding behavior."""
    print("Testing Conv2d padding...")

    # No padding: input unchanged
    conv_no_pad = Conv2d(1, 1, kernel_size=3, padding=0)
    x = np.ones((1, 1, 4, 4))
    result = conv_no_pad._apply_padding(x)
    assert result.shape == (1, 1, 4, 4), f"No-pad shape: expected (1,1,4,4), got {result.shape}"
    assert np.array_equal(result, x), "No-pad should return input unchanged"

    # Padding=1: adds 1 pixel border of zeros
    conv_pad1 = Conv2d(1, 1, kernel_size=3, padding=1)
    x = np.ones((1, 1, 3, 3))
    result = conv_pad1._apply_padding(x)
    assert result.shape == (1, 1, 5, 5), f"Pad-1 shape: expected (1,1,5,5), got {result.shape}"
    # Check that borders are zero
    assert np.all(result[:, :, 0, :] == 0), "Top border should be zeros"
    assert np.all(result[:, :, -1, :] == 0), "Bottom border should be zeros"
    assert np.all(result[:, :, :, 0] == 0), "Left border should be zeros"
    assert np.all(result[:, :, :, -1] == 0), "Right border should be zeros"
    # Check that center is preserved
    assert np.all(result[:, :, 1:4, 1:4] == 1), "Center should be preserved"

    # Padding=2: adds 2 pixel border
    conv_pad2 = Conv2d(1, 1, kernel_size=5, padding=2)
    x = np.ones((2, 3, 4, 4))
    result = conv_pad2._apply_padding(x)
    assert result.shape == (2, 3, 8, 8), f"Pad-2 shape: expected (2,3,8,8), got {result.shape}"
    # Batch and channel dims unchanged
    assert result.shape[0] == 2, "Batch dim should be unchanged"
    assert result.shape[1] == 3, "Channel dim should be unchanged"

    print("Conv2d padding works correctly!")

if __name__ == "__main__":
    test_unit_conv2d_padding()

Testing Conv2d padding...
Conv2d padding works correctly!


###  Unit Test: Conv2d Convolution Loops

In [20]:
def test_unit_conv2d_convolve_loops():
    """ Test unit : convolve loop"""
    print("Testing Conv2d convolve loops...")

    # Create a Conv2d with known weights (1 input channel, 1 output channel, 2x2 kernel)
    conv = Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)
    # Set weights to known values [[1, 0], [0, 1]]
    conv.weight = Tensor(np.array([[[[1.0, 0.0], 
                                      [0.0, 1.0]]]]), requires_grad=True)
    # Input: 1 batch, 1 channel, 3x3
    # [[1, 2, 3],
    #  [4, 5, 6],
    #  [7, 8, 9]]
    x = np.array([[[[1.0, 2.0, 3.0], 
                           [4.0, 5.0, 6.0],
                           [7.0, 8.0, 9.0]]]])
    # Output should be 2x2
    # pos(0,0) : 1*1+2*0+4*0+5*1 = 6
    # pos(0,1) : 2*1+3*0+5*0+6*1 = 8
    # pos(1,0) : 4*1+5*0+7*0+8*1 = 12
    # pos(1,1) : 5*1+6*0+8*0+9*1 = 14

    expected = np.array([[[[6.0, 8.0],
                        [12.0, 14.0]]]])
    output = conv._convolve_loops(x, batch_size=1, out_h=2, out_w=2)

    assert np.allclose(output, expected), f"Expected:\n{expected}\nGot:\n{output}"

    #Test with multiple output channels
    conv2 = Conv2d(in_channels=1, out_channels=2, kernel_size=2, bias=False)
    # Channel 0: all ones kernel, Channel 1: all twos kernel
    conv2.weight = Tensor(np.array([[[[1.0, 1.0], [1.0, 1.0]]],
                                     [[[2.0, 2.0], [2.0, 2.0]]]]), requires_grad=True)

    output2 = conv2._convolve_loops(x, batch_size=1, out_h=2, out_w=2)

    # Channel 0 (all-ones kernel): sum of each 2x2 window
    # pos(0,0): 1+2+4+5=12, pos(0,1): 2+3+5+6=16
    # pos(1,0): 4+5+7+8=24, pos(1,1): 5+6+8+9=28
    expected_ch0 = np.array([[12.0, 16.0], [24.0, 28.0]])
    expected_ch1 = expected_ch0 * 2  # All-twos kernel = 2x all-ones
    assert np.allclose(output2[0, 0], expected_ch0), f"Channel 0 mismatch"
    assert np.allclose(output2[0, 1], expected_ch1), f"Channel 1 mismatch"

    print("Conv2d convolution loops work correctly!")

if __name__ == "__main__":
    test_unit_conv2d_convolve_loops()
    


    

Testing Conv2d convolve loops...
Conv2d convolution loops work correctly!


###  Unit Test: Conv2d Forward (Composition)

In [26]:
def test_unit_conv2d():
    """Test Conv2d forward pass with multiple configurations."""
    print("Testing Conv2d...")

    # Test 1: Basic convolution without padding
    print("  Testing basic convolution...")
    conv1 = Conv2d(in_channels=3, out_channels=16, kernel_size=3)
    x1 = Tensor(rng.standard_normal((2, 3, 32, 32)))
    out1 = conv1(x1)

    expected_h = (32 - 3) + 1  # 30
    expected_w = (32 - 3) + 1  # 30
    assert out1.shape == (2, 16, expected_h, expected_w), f"Expected (2, 16, 30, 30), got {out1.shape}"

    # Test 2: Convolution with padding (same size)
    print("  Testing convolution with padding...")
    conv2 = Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1)
    x2 = Tensor(rng.standard_normal((1, 3, 28, 28)))
    out2 = conv2(x2)

    # With padding=1, output should be same size as input
    assert out2.shape == (1, 8, 28, 28), f"Expected (1, 8, 28, 28), got {out2.shape}"

    # Test 3: Convolution with stride
    print("  Testing convolution with stride...")
    conv3 = Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=2)
    x3 = Tensor(rng.standard_normal((1, 1, 16, 16)))
    out3 = conv3(x3)

    expected_h = (16 - 3) // 2 + 1  # 7
    expected_w = (16 - 3) // 2 + 1  # 7
    assert out3.shape == (1, 4, expected_h, expected_w), f"Expected (1, 4, 7, 7), got {out3.shape}"

    # Test 4: Parameter counting
    print("  Testing parameter counting...")
    conv4 = Conv2d(in_channels=64, out_channels=128, kernel_size=3, bias=True)
    params = conv4.parameters()

    # Weight: (128, 64, 3, 3) = 73,728 parameters
    # Bias: (128,) = 128 parameters
    # Total: 73,856 parameters
    weight_params = 128 * 64 * 3 * 3
    bias_params = 128
    total_params = weight_params + bias_params

    actual_weight_params = np.prod(conv4.weight.shape)
    actual_bias_params = np.prod(conv4.bias.shape) if conv4.bias is not None else 0
    actual_total = actual_weight_params + actual_bias_params

    assert actual_total == total_params, f"Expected {total_params} parameters, got {actual_total}"
    assert len(params) == 2, f"Expected 2 parameter tensors, got {len(params)}"

    # Test 5: No bias configuration
    print("  Testing no bias configuration...")
    conv5 = Conv2d(in_channels=3, out_channels=16, kernel_size=5, bias=False)
    params5 = conv5.parameters()
    assert len(params5) == 1, f"Expected 1 parameter tensor (no bias), got {len(params5)}"
    assert conv5.bias is None, "Bias should be None when bias=False"

    print("✅ Conv2d works correctly!")

if __name__ == "__main__":
    test_unit_conv2d()

Testing Conv2d...
  Testing basic convolution...
  Testing convolution with padding...
  Testing convolution with stride...
  Testing parameter counting...
  Testing no bias configuration...
✅ Conv2d works correctly!


##  Pooling Operations - Spatial Dimension Reduction

### MaxPool2d Implementation - Preserving Strong Features

MaxPool2d finds the strongest activation in each spatial window, creating a compressed representation that keeps the most important information.

In [95]:
#| export

class MaxPool2dBackward(Function):
    """ Gradien t computation for 2D max pooling """

    def __init__(self, x, output_shape, kernel_size, stride, padding):
        super().__init__(x)
        self.x = x
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        # Store max positions for gradient routing
        self.max_positions = {}

    def apply(self, grad_output):
        batch_size, channels, in_height, in_width = self.x.shape
        _,_, out_height, out_width = self.output_shape
        kernel_h, kernel_w = self.kernel_size

        if self.padding > 0:
            padded_input = np.pad(self.x.data,
                                 (0, 0), (0, 0),
                                 (self.padding, self.padding), (self.padding, self.padding),
                                  mode='constant', constant_values=-np.inf
                                 )
            grad_padded_input = np.zeros_like(padded_input)
        else:
            padded_input = self.x.data
            grad_padded_input = np.zeros_like(padded_input)

        for b in range(batch_size):
            for ch in range(channels):
                for out_h in range(out_height):
                    for out_w in range(out_width):
                        in_h_start = out_h * self.stride
                        in_w_start = out_w * self.stride

                        max_val = -np.inf
                        max_h, max_w = 0, 0
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                in_h = in_h_start + k_h
                                in_w = in_w_start + k_w
                                val = padded_input[b, ch, in_h, in_w]
                                
                                if val > max_val:
                                    max_val = val
                                    max_h, max_w = in_h, in_w
                                    
                        # Route gradient to max position
                        grad_padded_input[b, ch, max_h, max_w] += grad_output[b, ch, out_h, out_w]

        # Rmoving padding data
        if self.padding > 0:
            grad_input = grad_padded_input[:, :, self.padding:-self.padding, : self.padding:-self.padding]
        else:  
            grad_input = grad_padded_input

        return (grad_input,)

#| export

class MaxPool2d:
    """
    2D Max Pooling layer for spatial dimension reduction.
    """

    def __init__(self, kernel_size, stride=None, padding=0):
        """
        Initialize MaxPool2d layer.
        """
        
        # Handle kernel_size as int or tuple
        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size

        # Default stride equals kernel_size (non-overlapping)
        if stride is None:
            self.stride = self.kernel_size[0]
        else:
            self.stride = stride

        self.padding = padding
        

    def _compute_pool_output_shape(self, in_h, in_w):
        """
        Calculate output spatial dimensions for pooling.
        """
        
        kernel_h, kernel_w = self.kernel_size
        out_height = (in_h + 2 * self.padding - kernel_h) // self.stride + 1
        out_width = (in_w + 2 * self.padding - kernel_w) // self.stride + 1
        return out_height, out_width

    def _maxpool_loops(self, padded, batch_size, channels, out_h, out_w):
        """
        The core max pooling: find maximum value in each window.
        """
        
        kernel_h, kernel_w = self.kernel_size
        output = np.zeros((batch_size, channels, out_h, out_w))

        for b in range(batch_size):
            for c in range(channels):
                for oh in range(out_h):
                    for ow in range(out_w):
                        in_h_start = oh * self.stride
                        in_w_start = ow * self.stride

                        max_val = -np.inf
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                input_val = padded[b, c,
                                                  in_h_start + k_h,
                                                  in_w_start + k_w]
                                max_val = max(max_val, input_val)

                        output[b, c, oh, ow] = max_val

        return output
        

    def forward(self, x):
        """
        Forward pass through MaxPool2d layer.
        """
        
        # Step 1: Validate input
        validate_4d_input(x, "MaxPool2d")

        batch_size, channels, in_height, in_width = x.shape

        # Step 2: Compute output dimensions
        out_height, out_width = self._compute_pool_output_shape(in_height, in_width)

        # Step 3: Apply padding (use -inf for max pooling so padded values are never selected)
        if self.padding > 0:
            padded_input = np.pad(x.data,
                                ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)),
                                mode='constant', constant_values=-np.inf)
        else:
            padded_input = x.data

        # Step 4: Run max pooling loops
        output = self._maxpool_loops(padded_input, batch_size, channels, out_height, out_width)

        # Step 5: Return Tensor with gradient tracking
        result = Tensor(output, requires_grad=x.requires_grad)

        if result.requires_grad:
            result._grad_fn = MaxPool2dBackward(
                x, result.shape, self.kernel_size, self.stride, self.padding
            )

        return result
        

    def parameters(self):
        """Return empty list (pooling has no parameters)."""
        return []

    def __call__(self, x):
        """Enable model(x) syntax."""
        return self.forward(x)                             
            
        

### Unit Test: MaxPool2d Loops

This test validates that `_maxpool_loops` correctly finds the maximum
value in each pooling window.

In [28]:

def test_unit_maxpool2d_loops():
    """Test MaxPool2d._maxpool_loops with known values."""
    print("Testing MaxPool2d loops...")

    pool = MaxPool2d(kernel_size=2, stride=2)

    # Known 4x4 input
    padded = np.array([[[[1.0, 2.0, 3.0, 4.0],
                          [5.0, 6.0, 7.0, 8.0],
                          [9.0, 10.0, 11.0, 12.0],
                          [13.0, 14.0, 15.0, 16.0]]]])

    output = pool._maxpool_loops(padded, batch_size=1, channels=1, out_h=2, out_w=2)

    # Window maxes:
    # top-left: max(1,2,5,6) = 6
    # top-right: max(3,4,7,8) = 8
    # bottom-left: max(9,10,13,14) = 14
    # bottom-right: max(11,12,15,16) = 16
    expected = np.array([[[[6.0, 8.0], [14.0, 16.0]]]])
    assert np.allclose(output, expected), f"Expected:\n{expected}\nGot:\n{output}"

    # Test with negative values
    padded_neg = np.array([[[[-5.0, -1.0],
                              [-3.0, -2.0]]]])
    pool_small = MaxPool2d(kernel_size=2, stride=2)
    output_neg = pool_small._maxpool_loops(padded_neg, 1, 1, 1, 1)
    assert output_neg[0, 0, 0, 0] == -1.0, f"Max of negatives: expected -1.0, got {output_neg[0,0,0,0]}"

    print("MaxPool2d loops work correctly!")

if __name__ == "__main__":
    test_unit_maxpool2d_loops()

Testing MaxPool2d loops...
MaxPool2d loops work correctly!


### AvgPool2d Implementation - Smoothing and Generalizing Features

AvgPool2d computes the average of each spatial window, creating smoother features that are less sensitive to noise and exact pixel positions

In [31]:

#| export

class AvgPool2dBackward(Function):
    """
    Gradient computation for 2D average pooling.
    """

    def __init__(self, x, output_shape, kernel_size, stride, padding):
        super().__init__(x)
        self.x = x
        self.output_shape = output_shape
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def apply(self, grad_output):
        """
        Distribute each output gradient equally across its pooling window.
        """
        batch_size, channels, in_height, in_width = self.x.shape
        _, _, out_height, out_width = self.output_shape
        kernel_h, kernel_w = self.kernel_size
        kernel_area = kernel_h * kernel_w

        # Average pooling pads with zeros, so the gradient buffer is padded with
        # zeros too (matching the forward pass).
        if self.padding > 0:
            grad_input_padded = np.zeros(
                (batch_size, channels,
                 in_height + 2 * self.padding,
                 in_width + 2 * self.padding)
            )
        else:
            grad_input_padded = np.zeros_like(self.x.data)

        # Spread each output gradient equally over its window, accumulating overlaps.
        for b in range(batch_size):
            for c in range(channels):
                for out_h in range(out_height):
                    for out_w in range(out_width):
                        in_h_start = out_h * self.stride
                        in_w_start = out_w * self.stride
                        share = grad_output[b, c, out_h, out_w] / kernel_area
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                grad_input_padded[b, c, in_h_start + k_h, in_w_start + k_w] += share

        # Remove padding
        if self.padding > 0:
            grad_input = grad_input_padded[:, :,
                                          self.padding:-self.padding,
                                          self.padding:-self.padding]
        else:
            grad_input = grad_input_padded

        # Return as tuple (following Function protocol)
        return (grad_input,)

#| export

class AvgPool2d:
    """
    2D Average Pooling layer for spatial dimension reduction.
    """

    def __init__(self, kernel_size, stride=None, padding=0):
        """
        Initialize AvgPool2d layer.
        """
        
        # Handle kernel_size as int or tuple
        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size

        # Default stride equals kernel_size (non-overlapping)
        if stride is None:
            self.stride = self.kernel_size[0]
        else:
            self.stride = stride

        self.padding = padding
        

    def _compute_pool_output_shape(self, in_h, in_w):
        """
        Calculate output spatial dimensions for pooling.
        """
        
        kernel_h, kernel_w = self.kernel_size
        out_height = (in_h + 2 * self.padding - kernel_h) // self.stride + 1
        out_width = (in_w + 2 * self.padding - kernel_w) // self.stride + 1
        return out_height, out_width
        

    def _avgpool_loops(self, padded, batch_size, channels, out_h, out_w):
        """
        The core average pooling: compute mean of each window.
        """
        
        kernel_h, kernel_w = self.kernel_size
        output = np.zeros((batch_size, channels, out_h, out_w))

        for b in range(batch_size):
            for c in range(channels):
                for oh in range(out_h):
                    for ow in range(out_w):
                        in_h_start = oh * self.stride
                        in_w_start = ow * self.stride

                        window_sum = 0.0
                        for k_h in range(kernel_h):
                            for k_w in range(kernel_w):
                                input_val = padded[b, c,
                                                  in_h_start + k_h,
                                                  in_w_start + k_w]
                                window_sum += input_val

                        output[b, c, oh, ow] = window_sum / (kernel_h * kernel_w)

        return output
       

    def forward(self, x):
        """
        Forward pass through AvgPool2d layer.
        """
        
        # Step 1: Validate input
        validate_4d_input(x, "AvgPool2d")

        batch_size, channels, in_height, in_width = x.shape

        # Step 2: Compute output dimensions
        out_height, out_width = self._compute_pool_output_shape(in_height, in_width)

        # Step 3: Apply padding (use zeros for average pooling)
        if self.padding > 0:
            padded_input = np.pad(x.data,
                                ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)),
                                mode='constant', constant_values=0)
        else:
            padded_input = x.data

        # Step 4: Run average pooling loops
        output = self._avgpool_loops(padded_input, batch_size, channels, out_height, out_width)

        # Step 5: Return Tensor with gradient tracking
        result = Tensor(output, requires_grad=x.requires_grad)

        if result.requires_grad:
            result._grad_fn = AvgPool2dBackward(
                x, result.shape, self.kernel_size, self.stride, self.padding
            )

        return result
        

    def parameters(self):
        """Return empty list (pooling has no parameters)."""
        return []

    def __call__(self, x):
        """Enable model(x) syntax."""
        return self.forward(x)

### Unit Test: AvgPool2d Output Shape

This test validates that `_compute_pool_output_shape` correctly computes
the spatial dimensions after average pooling.

In [34]:


def test_unit_avgpool2d_output_shape():
    """Test AvgPool2d._compute_pool_output_shape."""
    print("Testing AvgPool2d output shape computation...")

    # Standard 2x2 pooling: halves dimensions
    pool = AvgPool2d(kernel_size=2, stride=2)
    oh, ow = pool._compute_pool_output_shape(8, 8)
    assert (oh, ow) == (4, 4), f"2x2 stride 2: expected (4, 4), got ({oh}, {ow})"

    # Non-square input
    oh, ow = pool._compute_pool_output_shape(16, 8)
    assert (oh, ow) == (8, 4), f"Non-square: expected (8, 4), got ({oh}, {ow})"

    # Overlapping pooling: kernel=3, stride=1
    pool_overlap = AvgPool2d(kernel_size=3, stride=1)
    oh, ow = pool_overlap._compute_pool_output_shape(5, 5)
    assert (oh, ow) == (3, 3), f"Overlapping: expected (3, 3), got ({oh}, {ow})"

    print("AvgPool2d output shape computation works correctly!")

if __name__ == "__main__":
    test_unit_avgpool2d_output_shape()

Testing AvgPool2d output shape computation...
AvgPool2d output shape computation works correctly!


### Unit Test: AvgPool2d Loops

This test validates that `_avgpool_loops` correctly computes the mean of

In [35]:


def test_unit_avgpool2d_loops():
    """Test AvgPool2d._avgpool_loops with known values."""
    print("Testing AvgPool2d loops...")

    pool = AvgPool2d(kernel_size=2, stride=2)

    # Known 4x4 input
    padded = np.array([[[[1.0, 2.0, 3.0, 4.0],
                          [5.0, 6.0, 7.0, 8.0],
                          [9.0, 10.0, 11.0, 12.0],
                          [13.0, 14.0, 15.0, 16.0]]]])

    output = pool._avgpool_loops(padded, batch_size=1, channels=1, out_h=2, out_w=2)

    # Window averages:
    # top-left: (1+2+5+6)/4 = 3.5
    # top-right: (3+4+7+8)/4 = 5.5
    # bottom-left: (9+10+13+14)/4 = 11.5
    # bottom-right: (11+12+15+16)/4 = 13.5
    expected = np.array([[[[3.5, 5.5], [11.5, 13.5]]]])
    assert np.allclose(output, expected), f"Expected:\n{expected}\nGot:\n{output}"

    # Test that avg is always <= max for same data
    pool_max = MaxPool2d(kernel_size=2, stride=2)
    max_output = pool_max._maxpool_loops(padded, 1, 1, 2, 2)
    assert np.all(output <= max_output), "Average should always be <= maximum"

    print("AvgPool2d loops work correctly!")

if __name__ == "__main__":
    test_unit_avgpool2d_loops()

Testing AvgPool2d loops...
AvgPool2d loops work correctly!


##  Batch Normalization - Stabilizing Deep Network Training

In [94]:
#| export

class BatchNorm2d:
    """ Compute batch normalization for 2D spatial inputs """

    def __init__(self, num_features, eps=1e-5, momentum=0.1):
    
        """ Initialize BatchNorm layer. """
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        # Learnable parameters (requires_grad=True for training)
        # gamma (scale): initialized to 1 so output = normalized input initially
        self.gamma = Tensor(np.ones(self.num_features), requires_grad=True)
        # beta (shift): initialized to 0 so no shift initially
        self.beta = Tensor(np.zeros(self.num_features), requires_grad=True)

        # Running statistics (not trained, accumulated during training)
        # These are used during evaluation for consistent normalization
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)

        # Training mode flag
        self.training = True

    def train(self):
        """Set layer to training mode."""
        self.training = True
        return self

    def eval(self):
        """ Set layer to evaluation mode """
        self.training = False
        return self

    def _validate_input(self, x):

        if len(x.shape) != 4:
            if len(x.shape) == 3:
                raise ValueError(
                    f"BatchNorm2d expected 4D input (batch, channels, height, width), got 3D: {x.shape}\n"
                    f"  ❌ Missing batch dimension\n"
                    f"  💡 BatchNorm2d computes statistics over the batch dimension\n"
                    f"  🔧 Add batch dim: x.reshape(1, {x.shape[0]}, {x.shape[1]}, {x.shape[2]})"
                )
            elif len(x.shape) == 2:
                raise ValueError(
                    f"BatchNorm2d expected 4D input (batch, channels, height, width), got 2D: {x.shape}\n"
                    f"  ❌ Got a matrix, expected an image tensor\n"
                    f"  💡 BatchNorm2d normalizes over spatial dimensions per channel\n"
                    f"  🔧 If this is a flattened image, reshape it: x.reshape(1, channels, height, width)"
                )
            else:
                raise ValueError(
                    f"BatchNorm2d expected 4D input (batch, channels, height, width), got {len(x.shape)}D: {x.shape}\n"
                    f"  ❌ Wrong number of dimensions\n"
                    f"  💡 BatchNorm2d expects: (batch_size, channels, height, width)\n"
                    f"  🔧 Reshape your input to 4D with the correct dimensions"
                )

        batch_size, channels, height, width = x.shape

        if channels != self.num_features:
            raise ValueError(
                f"BatchNorm2d channel mismatch: expected {self.num_features} channels, got {channels}\n"
                f"  ❌ Input has {channels} channels but BatchNorm2d was created for {self.num_features}\n"
                f"  💡 BatchNorm2d(num_features) must match the channel dimension of your input\n"
                f"  🔧 Either fix your input shape or create BatchNorm2d({channels})"
            )
            
    def _get_stats(self, x):
        """ Get nean and variance for normalization (Batch or running stats)"""
        
        if self.training:
            # Compute batch statistics per channel
            # Mean over batch and spatial dimensions: axes (0, 2, 3)
            batch_mean = np.mean(x.data, axis=(0, 2, 3))  # Shape: (C,)
            batch_var = np.var(x.data, axis=(0, 2, 3))    # Shape: (C,)

            # Update running statistics (exponential moving average)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var

            return batch_mean, batch_var
        else:
            # Use running statistics (frozen during eval)
            return self.running_mean, self.running_var

    def forward(self, x):
        """
        Forward pass through BatchNorm2d.
        """
        
        self._validate_input(x)

        batch_size, channels, height, width = x.shape
        mean, var = self._get_stats(x)

        # Normalize: (x - mean) / sqrt(var + eps)
        # Reshape mean and var for broadcasting: (C,) -> (1, C, 1, 1)
        mean_reshaped = mean.reshape(1, channels, 1, 1)
        var_reshaped = var.reshape(1, channels, 1, 1)

        x_normalized = (x.data - mean_reshaped) / np.sqrt(var_reshaped + self.eps)

        # Apply scale (gamma) and shift (beta)
        gamma_reshaped = self.gamma.data.reshape(1, channels, 1, 1)
        beta_reshaped = self.beta.data.reshape(1, channels, 1, 1)

        output = gamma_reshaped * x_normalized + beta_reshaped

        # Return Tensor with gradient tracking
        result = Tensor(output, requires_grad=x.requires_grad or self.gamma.requires_grad)

        return result
        

    def parameters(self):
        """Return learnable parameters (gamma and beta)."""
        return [self.gamma, self.beta]

    def __call__(self, x):
        """Enable model(x) syntax."""
        return self.forward(x)
        

###  Unit Test: BatchNorm2d._validate_input

In [41]:
def test_unit_batchnorm2d_validate_input():
    """ Test unit : BatchNorm2d_validate_input """
    print("Unit test : BatchNorm2d._validate_input...")

    bn = BatchNorm2d(num_features=16)
    x = Tensor(rng.standard_normal((2, 16, 8, 8)))

    bn._validate_input(x) # should pass silently

    # 3D input should raise
    try:
        bn._validate_input(Tensor(rng.standard_normal((16, 8, 8))))
        assert False, "Should have raised ValueError for 3D input"
    except ValueError as e:
        assert "3D" in str(e), f"Error should mention 3D, got: {e}"

    # Wrong channel count should raise
    try:
        bn._validate_input(Tensor(rng.standard_normal((2, 8, 4, 4))))
        assert False, "Should have raised ValueError for wrong channels"
    except ValueError as e:
        assert "mismatch" in str(e), f"Error should mention mismatch, got: {e}"

    print(" BatchNorm2d._validate_input works correctly!")

if __name__ == "__main__":
    test_unit_batchnorm2d_validate_input()
        

    

Unit test : BatchNorm2d._validate_input...
 BatchNorm2d._validate_input works correctly!


###  Unit Test: BatchNorm2d._get_stats

In [43]:
def test_unit_batchnorm2d_get_stats():
    """ Test BatchNorm2d._get_stats implementation."""
    print(" Unit Test: BatchNorm2d._get_stats...")

    bn = BatchNorm2d(num_features=4)
    x = Tensor(rng.standard_normal((8, 4, 6, 6)))

    # Training mode: should return batch stats and update running stats
    bn.train()
    running_mean_before = bn.running_mean.copy()
    mean, var = bn._get_stats(x)

    assert mean.shape == (4,), f"Expected per-channel mean shape (4,), got {mean.shape}"
    assert var.shape == (4,), f"Expected per-channel var shape (4,), got {var.shape}"
    assert not np.allclose(bn.running_mean, running_mean_before), \
        "Running mean should be updated in training mode"

    # Eval mode: should return running stats (frozen)
    bn.eval()
    running_mean_snapshot = bn.running_mean.copy()
    mean_eval, var_eval = bn._get_stats(x)

    assert np.allclose(mean_eval, running_mean_snapshot), \
        "Eval mode should return running mean"
    assert np.allclose(bn.running_mean, running_mean_snapshot), \
        "Running mean should not change in eval mode"

    print("BatchNorm2d._get_stats works correctly!")

if __name__ == "__main__":
    test_unit_batchnorm2d_get_stats()

 Unit Test: BatchNorm2d._get_stats...
BatchNorm2d._get_stats works correctly!


###  Unit Test: BatchNorm2d

This test validates batch normalization implementation.

In [45]:

def test_unit_batchnorm2d():
    """ Test BatchNorm2d implementation."""
    print(" Unit Test: BatchNorm2d...")

    # Test 1: Basic forward pass shape
    print("  Testing basic forward pass...")
    bn = BatchNorm2d(num_features=16)
    x = Tensor(rng.standard_normal((4, 16, 8, 8)))  # batch=4, channels=16, 8x8
    y = bn(x)

    assert y.shape == x.shape, f"Output shape should match input, got {y.shape}"

    # Test 2: Training mode normalization
    print("  Testing training mode normalization...")
    bn2 = BatchNorm2d(num_features=8)
    bn2.train()  # Ensure training mode

    # Create input with known statistics per channel
    x2 = Tensor(rng.standard_normal((32, 8, 4, 4)) * 10 + 5)  # Mean~5, std~10
    y2 = bn2(x2)

    # After normalization, each channel should have mean≈0, std≈1
    # (before gamma/beta are applied, since gamma=1, beta=0)
    for c in range(8):
        channel_mean = np.mean(y2.data[:, c, :, :])
        channel_std = np.std(y2.data[:, c, :, :])
        assert abs(channel_mean) < 0.1, f"Channel {c} mean should be ~0, got {channel_mean:.3f}"
        assert abs(channel_std - 1.0) < 0.1, f"Channel {c} std should be ~1, got {channel_std:.3f}"

    # Test 3: Running statistics update
    print("  Testing running statistics update...")
    initial_running_mean = bn2.running_mean.copy()
    print("Running mean :", initial_running_mean)
    # Forward pass updates running stats
    x3 = Tensor(rng.standard_normal((16, 8, 4, 4)) + 3)  # Offset mean
    _ = bn2(x3)

    # Running mean should have moved toward batch mean
    assert not np.allclose(bn2.running_mean, initial_running_mean), \
        "Running mean should update during training"

    # Test 4: Eval mode uses running statistics
    print("  Testing eval mode behavior...")
    bn3 = BatchNorm2d(num_features=4)

    # Train on some data to establish running stats
    for _ in range(10):
        x_train = Tensor(rng.standard_normal((8, 4, 4, 4)) * 2 + 1)
        _ = bn3(x_train)

    saved_running_mean = bn3.running_mean.copy()
    saved_running_var = bn3.running_var.copy()

    # Switch to eval mode
    bn3.eval()

    # Process different data - running stats should NOT change
    x_eval = Tensor(rng.standard_normal((2, 4, 4, 4)) * 5)  # Different distribution
    _ = bn3(x_eval)

    assert np.allclose(bn3.running_mean, saved_running_mean), \
        "Running mean should not change in eval mode"
    assert np.allclose(bn3.running_var, saved_running_var), \
        "Running var should not change in eval mode"

    # Test 5: Parameter counting
    print("  Testing parameter counting...")
    bn4 = BatchNorm2d(num_features=64)
    params = bn4.parameters()

    assert len(params) == 2, f"Should have 2 parameters (gamma, beta), got {len(params)}"
    assert params[0].shape == (64,), f"Gamma shape should be (64,), got {params[0].shape}"
    assert params[1].shape == (64,), f"Beta shape should be (64,), got {params[1].shape}"

    print("BatchNorm2d works correctly!")

if __name__ == "__main__":
    test_unit_batchnorm2d()

 Unit Test: BatchNorm2d...
  Testing basic forward pass...
  Testing training mode normalization...
  Testing running statistics update...
Running mean : [0.50891131 0.54838127 0.57182288 0.47781965 0.47285295 0.48844835
 0.4783189  0.50835216]
  Testing eval mode behavior...
  Testing parameter counting...
BatchNorm2d works correctly!


###  Unit Test: Pooling Operations

This test validates both max and average pooling implementations.

In [49]:


def test_unit_pooling():
    """ Test MaxPool2d and AvgPool2d implementations."""
    print(" Unit Test: Pooling Operations...")

    # Test 1: MaxPool2d basic functionality
    print("  Testing MaxPool2d...")
    maxpool = MaxPool2d(kernel_size=2, stride=2)
    x1 = Tensor(rng.standard_normal((1, 3, 8, 8)))
    out1 = maxpool(x1)

    expected_shape = (1, 3, 4, 4)  # 8/2 = 4
    assert out1.shape == expected_shape, f"MaxPool expected {expected_shape}, got {out1.shape}"

    # Test 2: AvgPool2d basic functionality
    print("  Testing AvgPool2d...")
    avgpool = AvgPool2d(kernel_size=2, stride=2)
    x2 = Tensor(rng.standard_normal((2, 16, 16, 16)))
    out2 = avgpool(x2)

    expected_shape = (2, 16, 8, 8)  # 16/2 = 8
    assert out2.shape == expected_shape, f"AvgPool expected {expected_shape}, got {out2.shape}"

    # Test 3: MaxPool vs AvgPool on known data
    print("  Testing max vs avg behavior...")
    # Create simple test case with known values
    test_data = np.array([[[[1, 2, 3, 4],
                           [5, 6, 7, 8],
                           [9, 10, 11, 12],
                           [13, 14, 15, 16]]]], dtype=np.float32)
    x3 = Tensor(test_data)

    maxpool_test = MaxPool2d(kernel_size=2, stride=2)
    avgpool_test = AvgPool2d(kernel_size=2, stride=2)

    max_out = maxpool_test(x3)
    avg_out = avgpool_test(x3)

    # For 2x2 windows:
    # Top-left: max([1,2,5,6]) = 6, avg = 3.5
    # Top-right: max([3,4,7,8]) = 8, avg = 5.5
    # Bottom-left: max([9,10,13,14]) = 14, avg = 11.5
    # Bottom-right: max([11,12,15,16]) = 16, avg = 13.5

    expected_max = np.array([[[[6, 8], [14, 16]]]])
    expected_avg = np.array([[[[3.5, 5.5], [11.5, 13.5]]]])

    assert np.allclose(max_out.data, expected_max), f"MaxPool values incorrect: {max_out.data} vs {expected_max}"
    assert np.allclose(avg_out.data, expected_avg), f"AvgPool values incorrect: {avg_out.data} vs {expected_avg}"

    # Test 4: Overlapping pooling (stride < kernel_size)
    print("  Testing overlapping pooling...")
    overlap_pool = MaxPool2d(kernel_size=3, stride=1)
    x4 = Tensor(rng.standard_normal((1, 1, 5, 5)))
    out4 = overlap_pool(x4)

    # Output: (5-3)/1 + 1 = 3
    expected_shape = (1, 1, 3, 3)
    assert out4.shape == expected_shape, f"Overlapping pool expected {expected_shape}, got {out4.shape}"

    # Test 5: No parameters in pooling layers
    print("  Testing parameter counts...")
    assert len(maxpool.parameters()) == 0, "MaxPool should have no parameters"
    assert len(avgpool.parameters()) == 0, "AvgPool should have no parameters"

    print(" Pooling operations work correctly!")

if __name__ == "__main__":
    test_unit_pooling()

 Unit Test: Pooling Operations...
  Testing MaxPool2d...
  Testing AvgPool2d...
  Testing max vs avg behavior...
  Testing overlapping pooling...
  Testing parameter counts...
 Pooling operations work correctly!


## Systems Analysis: Spatial Operation Performance

**computational complexity and memory trade-offs in spatial operations**

In [50]:


def analyze_convolution_complexity():
    """Analyze convolution computational complexity across different configurations."""
    print(" Analyzing Convolution Complexity...")

    # Test configurations optimized for educational demonstration (smaller sizes)
    configs = [
        {"input": (1, 3, 16, 16), "conv": (8, 3, 3), "name": "Small (16×16)"},
        {"input": (1, 3, 24, 24), "conv": (12, 3, 3), "name": "Medium (24×24)"},
        {"input": (1, 3, 32, 32), "conv": (16, 3, 3), "name": "Large (32×32)"},
        {"input": (1, 3, 16, 16), "conv": (8, 3, 5), "name": "Large Kernel (5×5)"},
    ]

    print(f"{'Configuration':<20} {'FLOPs':<15} {'Memory (MB)':<12} {'Time (ms)':<10}")
    print("-" * 70)

    for config in configs:
        # Create convolution layer
        in_ch = config["input"][1]
        out_ch, k_size = config["conv"][0], config["conv"][2]
        conv = Conv2d(in_ch, out_ch, kernel_size=k_size, padding=k_size//2)

        # Create input tensor
        x = Tensor(rng.standard_normal(config["input"]))

        # Calculate theoretical FLOPs
        batch, in_channels, h, w = config["input"]
        out_channels, kernel_size = config["conv"][0], config["conv"][2]

        # Each output element requires in_channels * kernel_size² multiply-adds
        flops_per_output = in_channels * kernel_size * kernel_size * 2  # 2 for MAC
        total_outputs = batch * out_channels * h * w  # Assuming same size with padding
        total_flops = flops_per_output * total_outputs

        # Measure memory usage
        input_memory = np.prod(config["input"]) * 4  # float32 = 4 bytes
        weight_memory = out_channels * in_channels * kernel_size * kernel_size * 4
        output_memory = batch * out_channels * h * w * 4
        total_memory = (input_memory + weight_memory + output_memory) / (1024 * 1024)  # MB

        # Measure execution time
        start_time = time.time()
        _ = conv(x)
        end_time = time.time()
        exec_time = (end_time - start_time) * 1000  # ms

        print(f"{config['name']:<20} {total_flops:<15,} {total_memory:<12.2f} {exec_time:<10.2f}")

    print("\n Key Insights:")
    print(" FLOPs scale as O(H×W×C_in×C_out×K²) - quadratic in spatial and kernel size")
    print(" Memory scales linearly with spatial dimensions and channels")
    print(" Large kernels dramatically increase computational cost")
    print(" This motivates more efficient convolution variants that reduce computational cost")

# Run the systems analysis
if __name__ == "__main__":
    analyze_convolution_complexity()

 Analyzing Convolution Complexity...
Configuration        FLOPs           Memory (MB)  Time (ms) 
----------------------------------------------------------------------
Small (16×16)        110,592         0.01         39.35     
Medium (24×24)       373,248         0.03         85.22     
Large (32×32)        884,736         0.08         147.70    
Large Kernel (5×5)   307,200         0.01         48.65     

 Key Insights:
 FLOPs scale as O(H×W×C_in×C_out×K²) - quadratic in spatial and kernel size
 Memory scales linearly with spatial dimensions and channels
 Large kernels dramatically increase computational cost
 This motivates more efficient convolution variants that reduce computational cost


In [52]:


def analyze_pooling_effects():
    """ Analyze pooling's impact on spatial dimensions and features."""
    print("\n Analyzing Pooling Effects...")

    # Create sample input with spatial structure
    # Simple edge pattern that pooling should preserve differently
    pattern = np.zeros((1, 1, 8, 8))
    pattern[0, 0, :, 3:5] = 1.0  # Vertical edge
    pattern[0, 0, 3:5, :] = 1.0  # Horizontal edge
    x = Tensor(pattern)

    print("Original 8×8 pattern:")
    print(x.data[0, 0])

    # Test different pooling strategies
    pools = [
        (MaxPool2d(2, stride=2), "MaxPool 2×2"),
        (AvgPool2d(2, stride=2), "AvgPool 2×2"),
        (MaxPool2d(4, stride=4), "MaxPool 4×4"),
        (AvgPool2d(4, stride=4), "AvgPool 4×4"),
    ]

    print(f"\n{'Operation':<15} {'Output Shape':<15} {'Feature Preservation'}")
    print("-" * 60)

    for pool_op, name in pools:
        result = pool_op(x)
        # Measure how much of the original pattern is preserved
        preservation = np.sum(result.data > 0.1) / np.prod(result.shape)
        print(f"{name:<15} {str(result.shape):<15} {preservation:<.2%}")

        print(f"  Output:")
        print(f"  {result.data[0, 0]}")
        print()

    print(" Key Insights:")
    print(" MaxPool preserves sharp features better (edge detection)")
    print(" AvgPool smooths features (noise reduction)")
    print(" Larger pooling windows lose more spatial detail")
    print(" Choice depends on task: classification vs detection vs segmentation")

# Run the systems analysis
if __name__ == "__main__":
    analyze_pooling_effects()


 Analyzing Pooling Effects...
Original 8×8 pattern:
[[0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0.]]

Operation       Output Shape    Feature Preservation
------------------------------------------------------------
MaxPool 2×2     (1, 1, 4, 4)    75.00%
  Output:
  [[0. 1. 1. 0.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [0. 1. 1. 0.]]

AvgPool 2×2     (1, 1, 4, 4)    75.00%
  Output:
  [[0.   0.5  0.5  0.  ]
 [0.5  0.75 0.75 0.5 ]
 [0.5  0.75 0.75 0.5 ]
 [0.   0.5  0.5  0.  ]]

MaxPool 4×4     (1, 1, 2, 2)    100.00%
  Output:
  [[1. 1.]
 [1. 1.]]

AvgPool 4×4     (1, 1, 2, 2)    100.00%
  Output:
  [[0.4375 0.4375]
 [0.4375 0.4375]]

 Key Insights:
 MaxPool preserves sharp features better (edge detection)
 AvgPool smooths features (noise reduction)
 Larger pooling windows lose more spatial detail
 Choice depends on task: classi

##  Integration - Building a Complete CNN

Now let's combine convolution and pooling into a complete CNN architecture. How spatial operations work together to transform raw pixels into meaningful features.

### SimpleCNN Implementation - Putting It All Together

In [80]:
#| export 

class SimpleCNN:
    """
        Simple CNN demonstrating spatial operations integration
    """

    def __init__(self, num_classes=10):
        """ Initialization of Simple CNN """
        
        self.conv1 = Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool1 = MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool2 = MaxPool2d(kernel_size=2, stride=2)
        
        # Final: 32 channels × 8×8 = 2048 features
        self.flattened_size = 32 * 8 * 8

        self.num_classes = num_classes
        #self.fc = Linear(self.flattened_size, self.num_classes)
    def forward(self, x):
        """ Forward pass through Simple CNN """

        # First conv block
        x = self.conv1(x)
        x = ReLU()(x)  # ReLU activation
        x = self.pool1(x)

        # Second conv block
        x = self.conv2(x)
        x = ReLU()(x)  # ReLU activation
        x = self.pool2(x)

        # Flatten for classification (reshape to 2D)
        batch_size = x.shape[0]
        x = x.reshape(batch_size, -1)
        #x = self.fc(x)
        # Return flattened features
        # In a complete implementation, this would go through a Linear layer
        return x

    def parameters(self):
        """Return all trainable parameters."""
        params = []
        params.extend(self.conv1.parameters())
        params.extend(self.conv2.parameters())
        #params.extend(self.fc.parameters())
        # Linear layer parameters would be added here
        return params

    def __call__(self, x):
        """Enable model(x) syntax."""
        return self.forward(x)

###  Unit Test: SimpleCNN Integration


In [85]:
def test_unit_simple_cnn():
    """ Test SimpleCNN integration with spatial operations."""
    print(" Unit Test: SimpleCNN Integration...")

    # Test 1: Forward pass with CIFAR-10 sized input
    print("  Testing forward pass...")
    model = SimpleCNN(num_classes=10)
    x = Tensor(rng.standard_normal((2, 3, 32, 32)))  # Batch of 2, RGB, 32×32

    features = model(x)
    linear = Linear(features.shape[1], model.num_classes)

    result = linear(features)
    # Expected: 2 samples, 32 channels × 8×8 spatial = 2048 features after conv2 and 2x10 after Linear 
    expected_shape = (2, 2048)
    assert features.shape == expected_shape, f"Expected {expected_shape}, got {features.shape}"

    # Test 2: Parameter counting
    print("  Testing parameter counting...")
    params = model.parameters()
    params.extend(linear.parameters())

    # Conv1: (16, 3, 3, 3) + bias (16,) = 432 + 16 = 448
    # Conv2: (32, 16, 3, 3) + bias (32,) = 4608 + 32 = 4640
    # fc: (2048, 10) + bias (10,) = 20490
    # Total: 448 + 4640 + 20490 = 25578 parameters

    conv1_params = 16 * 3 * 3 * 3 + 16  # weights + bias
    conv2_params = 32 * 16 * 3 * 3 + 32  # weights + bias
    fc_params = 2048 * 10 + 10  
    expected_total = conv1_params + conv2_params + fc_params

    actual_total = sum(np.prod(p.shape) for p in params)
    assert actual_total == expected_total, f"Expected {expected_total} parameters, got {actual_total}"

    # Test 3: Different input sizes
    print("  Testing different input sizes...")

    # Test with different spatial dimensions
    x_small = Tensor(rng.standard_normal((1, 3, 16, 16)))
    features_small = model(x_small)

    # 16×16 → 8×8 → 4×4, so 32 × 4×4 = 512 features
    expected_small = (1, 512)
    assert features_small.shape == expected_small, f"Expected {expected_small}, got {features_small.shape}"

    # Test 4: Batch processing
    print("  Testing batch processing...")
    x_batch = Tensor(rng.standard_normal((8, 3, 32, 32)))
    features_batch = model(x_batch)

    expected_batch = (8, 2048)
    assert features_batch.shape == expected_batch, f"Expected {expected_batch}, got {features_batch.shape}"

    print("SimpleCNN integration works correctly!")

if __name__ == "__main__":
    test_unit_simple_cnn()

 Unit Test: SimpleCNN Integration...
  Testing forward pass...
  Testing parameter counting...
  Testing different input sizes...
  Testing batch processing...
SimpleCNN integration works correctly!


##  Module Integration Test

Final validation that everything works together correctly.

In [89]:


def test_module():
    """ Module Test: Complete Integration
    """
    print(" RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")

    # Conv2d helper tests
    test_unit_conv2d_output_shape()
    test_unit_conv2d_padding()
    test_unit_conv2d_convolve_loops()
    test_unit_conv2d()

    # BatchNorm2d helper tests
    test_unit_batchnorm2d_validate_input()
    test_unit_batchnorm2d_get_stats()

    # MaxPool2d helper tests
    test_unit_maxpool2d_output_shape()
    test_unit_maxpool2d_loops()

    # AvgPool2d helper tests
    test_unit_avgpool2d_output_shape()
    test_unit_avgpool2d_loops()

    # Remaining unit tests
    test_unit_batchnorm2d()
    test_unit_pooling()
    test_unit_simple_cnn()

    print("\nRunning integration scenarios...")

    # Test realistic CNN workflow with BatchNorm
    print(" Integration Test: Complete CNN pipeline with BatchNorm...")

    # Create a mini CNN for CIFAR-10 with BatchNorm (modern architecture)
    conv1 = Conv2d(3, 8, kernel_size=3, padding=1)
    bn1 = BatchNorm2d(8)
    pool1 = MaxPool2d(2, stride=2)
    conv2 = Conv2d(8, 16, kernel_size=3, padding=1)
    bn2 = BatchNorm2d(16)
    pool2 = AvgPool2d(2, stride=2)

    # Process batch of images (training mode)
    batch_images = Tensor(rng.standard_normal((4, 3, 32, 32)))

    # Forward pass: Conv → BatchNorm → ReLU → Pool (modern pattern)
    x = conv1(batch_images)  # (4, 8, 32, 32)
    x = bn1(x)               # (4, 8, 32, 32) - normalized
    x = Tensor(np.maximum(0, x.data))  # ReLU
    x = pool1(x)             # (4, 8, 16, 16)

    x = conv2(x)             # (4, 16, 16, 16)
    x = bn2(x)               # (4, 16, 16, 16) - normalized
    x = Tensor(np.maximum(0, x.data))  # ReLU
    features = pool2(x)      # (4, 16, 8, 8)

    # Validate shapes at each step
    assert features.shape[0] == 4, f"Batch size should be preserved, got {features.shape[0]}"
    assert features.shape == (4, 16, 8, 8), f"Final features shape incorrect: {features.shape}"

    # Test parameter collection across all layers
    all_params = []
    all_params.extend(conv1.parameters())
    all_params.extend(bn1.parameters())
    all_params.extend(conv2.parameters())
    all_params.extend(bn2.parameters())

    # Pooling has no parameters
    assert len(pool1.parameters()) == 0
    assert len(pool2.parameters()) == 0

    # BatchNorm has 2 params each (gamma, beta)
    assert len(bn1.parameters()) == 2, f"BatchNorm should have 2 parameters, got {len(bn1.parameters())}"

    # Total: Conv1 (2) + BN1 (2) + Conv2 (2) + BN2 (2) = 8 parameters
    assert len(all_params) == 8, f"Expected 8 parameter tensors total, got {len(all_params)}"

    # Test train/eval mode switching
    print("Integration Test: Train/Eval mode switching...")
    bn1.eval()
    bn2.eval()

    # Run inference with single sample (would fail with batch stats)
    single_image = Tensor(rng.standard_normal((1, 3, 32, 32)))
    x = conv1(single_image)
    x = bn1(x)  # Uses running stats, not batch stats
    assert x.shape == (1, 8, 32, 32), f"Single sample inference should work in eval mode"

    print(" CNN pipeline with BatchNorm works correctly!")

    # Test memory efficiency comparison
    print(" Integration Test: Memory efficiency analysis...")

    # Compare different pooling strategies (reduced size for faster execution)
    input_data = Tensor(rng.standard_normal((1, 16, 32, 32)))

    # No pooling: maintain spatial size
    conv_only = Conv2d(16, 32, kernel_size=3, padding=1)
    no_pool_out = conv_only(input_data)
    no_pool_size = np.prod(no_pool_out.shape) * 4  # float32 bytes

    # With pooling: reduce spatial size
    conv_with_pool = Conv2d(16, 32, kernel_size=3, padding=1)
    pool = MaxPool2d(2, stride=2)
    pool_out = pool(conv_with_pool(input_data))
    pool_size = np.prod(pool_out.shape) * 4  # float32 bytes

    memory_reduction = no_pool_size / pool_size
    assert memory_reduction == 4.0, f"2×2 pooling should give 4× memory reduction, got {memory_reduction:.1f}×"

    print(f"  Memory reduction with pooling: {memory_reduction:.1f}×")
    print(" Memory efficiency analysis complete!")

    print("\n" + "=" * 50)

if __name__ == "__main__":
    test_module()



 RUNNING MODULE INTEGRATION TEST
Running unit tests...
Testing Conv2d output shape computation...
Conv2d output shape computation works correctly!
Testing Conv2d padding...
Conv2d padding works correctly!
Testing Conv2d convolve loops...
Conv2d convolution loops work correctly!
Testing Conv2d...
  Testing basic convolution...
  Testing convolution with padding...
  Testing convolution with stride...
  Testing parameter counting...
  Testing no bias configuration...
✅ Conv2d works correctly!
Unit test : BatchNorm2d._validate_input...
 BatchNorm2d._validate_input works correctly!
 Unit Test: BatchNorm2d._get_stats...
BatchNorm2d._get_stats works correctly!
Testing MaxPool2d output shape computation...
MaxPool2d output shape computation works correctly!
Testing MaxPool2d loops...
MaxPool2d loops work correctly!
Testing AvgPool2d output shape computation...
AvgPool2d output shape computation works correctly!
Testing AvgPool2d loops...
AvgPool2d loops work correctly!
 Unit Test: BatchNorm2d

In [96]:
from nbdev.export import nb_export
nb_export('convolutions.ipynb', lib_path='../tinytorch/')